<a href="https://colab.research.google.com/github/mesuvein23/SocrateAI/blob/master/agent2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q google-adk

In [24]:
import os
import datetime

from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, ToolContext
from google.adk.models import Gemini
from google.genai import types
from google.colab import userdata

# ── env/config ───────────────────────────────────────────────────────────────
# ✏️ Colab has no .env file, so the key comes from Colab Secrets instead
os.environ["GOOGLE_API_KEY"] = userdata.get("agentkey")

# ✏️ Retry automatically when Gemini says "503 high demand"
MODEL = Gemini(
    model="gemini-3.6-flash",   # if it's always busy, try "gemini-flash-lite-latest"
    retry_options=types.HttpRetryOptions(
        attempts=5, initial_delay=2, http_status_codes=[429, 500, 502, 503, 504]
    ),
)

# ✏️ New tool: lets a checker stop its loop when the work is good
def exit_loop(tool_context: ToolContext):
    """Call this ONLY when the content passes validation."""
    tool_context.actions.escalate = True
    return {}

# ── Sub-Agent: Planner ───────────────────────────────────────────────────────
blog_planner = Agent(
    name="BlogPlanner",
    model=MODEL,
    description="Creates a practical, skimmable outline in Markdown.",
    instruction="""
You are a technical content strategist. Produce a clear Markdown outline with:
- Title
- Short intro
- 4–6 main sections (each with 2–3 bullets)
- Conclusion

Codebase context (may be empty): {codebase_context?}
Feedback from last attempt (may be empty): {outline_feedback?}

Return only the outline in Markdown.
""",  # ✏️ {placeholders} so the agent actually sees state values
    output_key="blog_outline",
)

class OutlineValidationChecker(Agent):
    def __init__(self):
        super().__init__(
            name="OutlineValidationChecker",
            model=MODEL,
            description="Validates that the outline is usable.",
            instruction="""
Check this outline:

{blog_outline}

If it has a title, intro, 4–6 sections, and a conclusion, call the exit_loop tool.
Otherwise respond "retry" and list missing pieces.
""",
            tools=[exit_loop],              # ✏️ added
            output_key="outline_feedback",  # ✏️ was "validation_result"
        )

robust_blog_planner = LoopAgent(
    name="RobustBlogPlanner",
    description="Creates a blog outline for a topic, retrying if validation fails.",
    sub_agents=[blog_planner, OutlineValidationChecker()],
    max_iterations=3,
)

# ── Sub-Agent: Writer ────────────────────────────────────────────────────────
blog_writer = Agent(
    name="BlogWriter",
    model=MODEL,
    description="Writes a technical blog post from the outline.",
    instruction="""
Write a complete Markdown article from this outline:

{blog_outline}

Feedback from last attempt (may be empty): {post_feedback?}

Guidelines:
- Audience: software engineers; skip basics and focus on practical insight.
- Explain both the 'how' and 'why'.
- Include concise code snippets when helpful.
- Follow the outline's structure (H2/H3).
- Output only the final article in Markdown (no fence around the whole post).
""",
    output_key="blog_post",
)

class BlogPostValidationChecker(Agent):
    def __init__(self):
        super().__init__(
            name="BlogPostValidationChecker",
            model=MODEL,
            description="Validates the final post.",
            instruction="""
Check this blog post:

{blog_post}

Look for: intro, clear sections matching the outline, conclusion, and technical clarity.
If it passes, call the exit_loop tool. Else respond "retry" with the specific fixes.
""",
            tools=[exit_loop],           # ✏️ added
            output_key="post_feedback",  # ✏️ was "validation_result"
        )

robust_blog_writer = LoopAgent(
    name="RobustBlogWriter",
    description="Writes the full blog post from the saved outline, retrying if validation fails.",
    sub_agents=[blog_writer, BlogPostValidationChecker()],
    max_iterations=3,
)

# Expose planner/writer as tools so the root agent can call them explicitly
planner_tool = agent_tool.AgentTool(agent=robust_blog_planner)
writer_tool = agent_tool.AgentTool(agent=robust_blog_writer)

# ── Root Agent: Plan → Write ─────────────────────────────────────────────────
root_agent = Agent(
    name="Blogger",
    model=MODEL,
    description="Minimal multi-agent blogger that plans and writes.",
    instruction=f"""
If the user gives a topic:
1) Call the planner tool to generate the outline.
2) Call the writer tool to produce the full draft.
3) End with 3 alternate titles and 2 tweet-length hooks.

Date: {datetime.datetime.now().strftime("%Y-%m-%d")}
""",
    tools=[planner_tool, writer_tool],
)

print("✅ Agents ready")

✅ Agents ready


/tmp/ipykernel_7680/2480303185.py:66: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  robust_blog_planner = LoopAgent(
/tmp/ipykernel_7680/2480303185.py:113: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  robust_blog_writer = LoopAgent(


In [25]:
from google.adk.runners import InMemoryRunner

TOPIC = "Python asyncio for beginners"   # 👈 change your topic here

runner = InMemoryRunner(agent=root_agent, app_name="blogger")
session = await runner.session_service.create_session(app_name="blogger", user_id="me")
msg = types.Content(role="user", parts=[types.Part(text=TOPIC)])

async for event in runner.run_async(user_id="me", session_id=session.id, new_message=msg):
    if event.is_final_response() and event.content and event.content.parts:
        print(event.content.parts[0].text)

s = await runner.session_service.get_session(app_name="blogger", user_id="me", session_id=session.id)
print("\n========== FULL ARTICLE ==========\n")
print(s.state.get("blog_post"))

# Mastering Python Asyncio: A Beginner's Practical Guide

Python is renowned for its simple syntax and readable code, but when it comes to performing many tasks at once—like fetching data from hundreds of web pages, querying multiple databases, or handling thousands of API requests—traditional synchronous code can quickly become a performance bottleneck. 

Enter **`asyncio`**, Python's built-in library for writing concurrent code using the `async`/`await` syntax. If you’ve ever found `asyncio` intimidating or confusing, this guide is designed for you. We will break down the core concepts step-by-step so you can start writing fast, non-blocking Python applications today.

---

## 1. Synchronous vs. Asynchronous Programming

To understand why `asyncio` is useful, let's contrast synchronous and asynchronous execution:

* **Synchronous (Sync):** Your program executes code sequentially, line by line. If a line of code initiates an I/O operation—such as reading a file from disk or downloadin